# BIOT: frozen encoder and linear classifier
Binary FACED; 18 bipolar channels, five-second windows, fold 0 and a train-only linear classifier.

In [ ]:
import os,sys,json,hashlib,time,random
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from scipy.signal import resample_poly
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02')
source=Path(r'D:\EEG Experiment Artifacts\2026-11-02\pretrained_experiments\dependencies')
weights=Path(r'D:\EEG Experiment Artifacts\pretrained_models\BIOT\weights\model.safetensors')
output=root/'pretrained_experiments'/'biot_frozen';output.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(source))
from braindecode.models import BIOT
import zipfile,pickle
seed=42;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
torch.set_num_threads(4);device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type=='cuda','GPU is required for the configured screening run.'
protocol=json.loads((root/'binary_de_psd'/'protocol.json').read_text(encoding='utf8'));split=protocol['splits'][0]
subjects=np.repeat(np.arange(123),24);video_ids=np.r_[0:12,16:28];labels=np.tile(np.r_[np.zeros(12,dtype=np.int64),np.ones(12,dtype=np.int64)],123)
indices={k:np.flatnonzero(np.isin(subjects,split[k])) for k in ('train','val')}
assert not set(split['train'])&set(split['val']) and not set(split['test'])&set(split['train']+split['val'])
raw_file=root/'cache'/'raw_windows_30ch.npy';raw=np.load(raw_file,mmap_mode='r');assert raw.shape==(3444,30,30,250)
def sha(path):
 h=hashlib.sha256()
 with path.open('rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
manifest={'model':'BIOT six-dataset 18-channel checkpoint, frozen','weight_sha256':sha(weights),'data_zip_sha256':sha(Path(r'C:\FACED dataset\Other Files\Processed_data.zip')),
 'fold':0,'seed':seed,'epochs':10,'split':split,'preprocessing':'original 32-channel FACED signals; reconstruct 18 bipolar channels, per-trial/channel z-score, resample 250->200 Hz, six 5 s windows',
 'features':'BIOT 256-dimensional encoder embedding per 5 s window','selection':'validation balanced accuracy','gpu':torch.cuda.get_device_name(0)}
manifest_file=output/'config.json'
if manifest_file.exists():assert json.loads(manifest_file.read_text())==manifest
else:manifest_file.write_text(json.dumps(manifest,indent=2),encoding='utf8')
encoder=BIOT.from_pretrained(str(weights.parent),local_files_only=True).encoder.to(device).eval()
for param in encoder.parameters():param.requires_grad_(False)
print('DEVICE',device,manifest['gpu'],'TRAIN TRIALS',len(indices['train']),'VALIDATION TRIALS',len(indices['val']),flush=True)


In [ ]:
# Cache embeddings by trial and safely resume from the last completed trial.
zip_path=Path(r'C:\FACED dataset\Other Files\Processed_data.zip')
channels=protocol['settings']['channels']+['A2','A1']
channel_index={n.upper():i for i,n in enumerate(channels)}
bipolar_pairs=[('FP1','F7'),('F7','T7'),('T7','P7'),('P7','O1'),('FP2','F8'),('F8','T8'),('T8','P8'),('P8','O2'),('FP1','F3'),('F3','C3'),('C3','P3'),('P3','O1'),('FP2','F4'),('F4','C4'),('C4','P4'),('P4','O2'),('C3','A2'),('C4','A1')]
loaded_subject=-1
start_all=time.perf_counter()
for split_name in ['train','val']:
    trial_ids=indices[split_name];file=output/(split_name+'_embeddings.npy');progress=output/(split_name+'_progress.json')
    if file.exists():
        emb=np.lib.format.open_memmap(file,mode='r+');done=json.loads(progress.read_text())['completed_trials']
        assert emb.shape==(len(trial_ids),6,256)
    else:
        emb=np.lib.format.open_memmap(file,mode='w+',dtype=np.float32,shape=(len(trial_ids),6,256));done=0
        progress.write_text(json.dumps({'completed_trials':0}),encoding='utf8')
    timer=time.perf_counter();count=0
    with torch.inference_mode():
        for i in range(done,len(trial_ids)):
            j=int(trial_ids[i]);sub=int(subjects[j]);video=int(video_ids[j%24])
            if sub!=loaded_subject:
                with zipfile.ZipFile(zip_path) as archive:
                    subject_data=np.asarray(pickle.loads(archive.read(f'Processed_data/sub{sub:03d}.pkl')),dtype=np.float32)
                assert subject_data.shape==(28,32,7500);loaded_subject=sub
            x=subject_data[video]
            x=np.stack([x[channel_index[a]]-x[channel_index[b]] for a,b in bipolar_pairs])
            x=(x-x.mean(axis=-1,keepdims=True))/np.maximum(x.std(axis=-1,keepdims=True),1e-6)
            x=resample_poly(x,4,5,axis=-1).reshape(18,6,1000).transpose(1,0,2).copy()
            tensor=torch.from_numpy(x).to(device)
            result=encoder(tensor);assert result.shape==(6,256)
            result=result.cpu().numpy()
            emb[i]=result;count+=1
            if count==3:
                elapsed=(time.perf_counter()-timer)/count;remaining=sum(len(indices[k]) for k in ['train','val'])-i-1 if split_name=='train' else len(trial_ids)-i-1
                print('PROFILE_SECONDS_PER_TRIAL',round(elapsed,3),'ESTIMATED_REMAINING_MINUTES',round(elapsed*remaining/60,1),flush=True)
            if (i+1)%10==0 or i+1==len(trial_ids):
                emb.flush();tmp=progress.with_suffix('.tmp');tmp.write_text(json.dumps({'completed_trials':i+1}),encoding='utf8');tmp.replace(progress)
                print('EMBEDDINGS',split_name,i+1,'/',len(trial_ids),flush=True)
    del emb
print('EMBEDDING_EXTRACTION_SECONDS',round(time.perf_counter()-start_all,1),flush=True)


In [ ]:
# Fit normalization only on the training split, then train a two-class linear head.
train=np.load(output/'train_embeddings.npy',mmap_mode='r').reshape(-1,256)
val=np.load(output/'val_embeddings.npy',mmap_mode='r').reshape(-1,256)
y_train=np.repeat(labels[indices['train']],6);y_val=np.repeat(labels[indices['val']],6)
assert len(train)==len(y_train) and len(val)==len(y_val)
scaler_path=output/'scaler.npz'
if scaler_path.exists():
 s=np.load(scaler_path);mean=s['mean'];std=s['std']
else:
 total=np.zeros(256,dtype=np.float64);total2=np.zeros(256,dtype=np.float64)
 for start in range(0,len(train),2048):
  x=np.asarray(train[start:start+2048],dtype=np.float64);total+=x.sum(0);total2+=(x*x).sum(0)
 mean=total/len(train);std=np.sqrt(np.maximum(total2/len(train)-mean*mean,1e-12))
 mean=mean.astype(np.float32);std=np.maximum(std,1e-6).astype(np.float32);np.savez(scaler_path,mean=mean,std=std)
model=nn.Linear(256,2).to(device);optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-3)
best=-1;history=[];rng=np.random.default_rng(seed)
for epoch in range(1,11):
 model.train();order=rng.permutation(len(train));losses=[]
 for batch in np.array_split(order,int(np.ceil(len(order)/512))):
  x=torch.from_numpy((np.asarray(train[batch])-mean)/std).to(device);y=torch.from_numpy(y_train[batch]).to(device)
  optimizer.zero_grad(set_to_none=True);loss=nn.functional.cross_entropy(model(x),y);loss.backward();optimizer.step();losses.append(loss.item())
 model.eval();parts=[]
 with torch.inference_mode():
  for start in range(0,len(val),512):
   x=torch.from_numpy((np.asarray(val[start:start+512])-mean)/std).to(device);parts.append(model(x).cpu().numpy())
 logits=np.concatenate(parts);pred=logits.argmax(1);score=balanced_accuracy_score(y_val,pred)
 history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'validation_accuracy':float(accuracy_score(y_val,pred)),'validation_balanced_accuracy':float(score)})
 pd.DataFrame(history).to_csv(output/'history.csv',index=False)
 if score>best:
  best=score;torch.save({'state_dict':model.state_dict(),'epoch':epoch},output/'best_model.pt')
  np.savez_compressed(output/'validation_predictions.npz',logits=logits.reshape(-1,6,2),predictions=pred.reshape(-1,6),labels=y_val.reshape(-1,6),subjects=subjects[indices['val']])
 print('HEAD EPOCH',epoch,'VALIDATION BALANCED ACCURACY',round(score,5),flush=True)
result={'best_epoch':int(pd.DataFrame(history).iloc[np.argmax([r['validation_balanced_accuracy'] for r in history])]['epoch']),
        'validation_balanced_accuracy':float(best),'protocol':'one subject-disjoint fold, checkpoint selected on validation',
        'train_subjects':len(split['train']),'validation_subjects':len(split['val'])}
(output/'metrics.json').write_text(json.dumps(result,indent=2),encoding='utf8');print('COMPLETE',json.dumps(result),flush=True)
